# HÂKİM — PaddleOCR on Colab A100

Local CPU is ~4 min/page. A100 should finish ~700 pages in minutes–tens of minutes.

**Runtime → Change runtime type → GPU → A100**

1. Copy PC folder `data2/` (the 9 law PDFs) to Drive: `MyDrive/hakim-ocr/data2/`
2. Run all cells.
3. Results: `MyDrive/hakim-ocr/pdf_ocr_work/`
4. On PC: copy that folder to `data/raw/pdf_ocr_work/` then:

```powershell
uv run python scripts/ingest_data2_pdfs.py --from-ocr-work
```

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU runtime seçili değil"
print(torch.cuda.get_device_name(0))
print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

%pip install -q pymupdf pillow numpy
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q paddleocr

import paddle
print("paddle", paddle.__version__, "cuda", paddle.device.is_compiled_with_cuda())

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

IN_DIR = Path("/content/drive/MyDrive/hakim-ocr/data2")
OUT_ROOT = Path("/content/drive/MyDrive/hakim-ocr/pdf_ocr_work")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

pdfs = sorted(IN_DIR.glob("*.pdf"), key=lambda p: p.stat().st_size)
print("PDFs", len(pdfs), "from", IN_DIR)
for p in pdfs:
    print(f"  {p.name:40} {p.stat().st_size/1024:.0f} KB")
assert pdfs, f"Drive'a koy: {IN_DIR}  (data2/*.pdf)"

In [ ]:
import json, re, time, sys
import numpy as np
import pymupdf
from PIL import Image
from paddleocr import PaddleOCR

def slug(name: str) -> str:
    stem = Path(name).stem.lower()
    for a, b in ("ı", "i"), ("İ", "i"), ("ş", "s"), ("ğ", "g"), ("ü", "u"), ("ö", "o"), ("ç", "c"):
        stem = stem.replace(a, b)
    return re.sub(r"[^a-z0-9]+", "-", stem).strip("-") or "doc"

def lines_from_result(result):
    lines = []
    if result is None:
        return lines
    if isinstance(result, dict):
        for t in result.get("rec_texts") or result.get("texts") or []:
            if t:
                lines.append(str(t))
        return lines
    if hasattr(result, "get") and callable(result.get):
        texts = result.get("rec_texts") or result.get("texts")
        if texts:
            return [str(t) for t in texts if t]
    if hasattr(result, "rec_texts"):
        return [str(t) for t in (result.rec_texts or []) if t]
    if isinstance(result, list):
        for item in result:
            lines.extend(lines_from_result(item) if not isinstance(item, (list, tuple, str)) else [])
            if isinstance(item, list):
                for row in item:
                    if isinstance(row, (list, tuple)) and len(row) >= 2:
                        payload = row[1]
                        if isinstance(payload, (list, tuple)) and payload:
                            lines.append(str(payload[0]))
                        elif isinstance(payload, str):
                            lines.append(payload)
                    elif isinstance(row, str):
                        lines.append(row)
            elif isinstance(item, str):
                lines.append(item)
    return lines

def page_image(doc, index, dpi):
    zoom = dpi / 72.0
    pix = doc.load_page(index).get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False)
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

print("helpers ok")

In [ ]:
ocr = PaddleOCR(
    lang="en",
    device="gpu:0",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
print("PaddleOCR ready on GPU")

In [ ]:
DPI = 180
SKIP_DONE = True  # Drive'da content.txt + done.json varsa atla

def ocr_one(pdf: Path, work: Path) -> None:
    work.mkdir(parents=True, exist_ok=True)
    content = work / "content.txt"
    done = work / "done.json"
    partial = work / "partial.txt"
    progress = work / "progress.json"
    if SKIP_DONE and done.exists() and content.exists() and content.stat().st_size > 200:
        print(f"[skip] {pdf.name}")
        return
    start = 0
    if progress.exists():
        start = int(json.loads(progress.read_text()).get("done_pages") or 0)
    doc = pymupdf.open(pdf)
    total = len(doc)
    print(f"[ocr] {pdf.name} {start+1}-{total}/{total} dpi={DPI}", flush=True)
    t0 = time.time()
    try:
        for i in range(start, total):
            page_text = "\n".join(lines_from_result(ocr.predict(np.array(page_image(doc, i, DPI))))).strip()
            with partial.open("a", encoding="utf-8") as fh:
                if i or start:
                    fh.write("\n\n")
                fh.write(page_text)
            elapsed = time.time() - t0
            eta = (elapsed / (i - start + 1)) * (total - i - 1)
            print(f"  page {i+1}/{total} chars={len(page_text)} elapsed={elapsed:.0f}s eta={eta:.0f}s", flush=True)
            progress.write_text(json.dumps({"done_pages": i + 1, "pages_total": total}, ensure_ascii=False, indent=2))
    finally:
        doc.close()
    text = partial.read_text(encoding="utf-8").strip() if partial.exists() else ""
    content.write_text(text, encoding="utf-8")
    done.write_text(json.dumps({"source": pdf.name, "pages_total": total, "chars": len(text), "engine": "paddleocr-gpu"}, ensure_ascii=False, indent=2))
    print(f"[done] {pdf.name} chars={len(text)}", flush=True)

for pdf in pdfs:
    ocr_one(pdf, OUT_ROOT / slug(pdf.name))
print("ALL DONE", OUT_ROOT)

In [ ]:
import shutil
zip_path = Path("/content/drive/MyDrive/hakim-ocr/pdf_ocr_work.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT_ROOT)
print("zip", zip_path, "MB", round(zip_path.stat().st_size / 1e6, 2))
print("PC'ye indir: Drive → hakim-ocr/pdf_ocr_work.zip")
print("Aç: data/raw/pdf_ocr_work/")
print("Sonra: uv run python scripts/ingest_data2_pdfs.py --from-ocr-work")